# Infrastructure Setup

In [1]:
pip install --no-cache-dir -r requirements.txt


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Imports & Setup

In [2]:
import os
from typing import List, Dict, Any, Optional
from io import BytesIO
import re
import requests
import arxiv
import PyPDF2
from dotenv import load_dotenv
from langchain.agents.agent import AgentExecutor, AgentAction, AgentFinish
from langchain.agents import create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import Tool, StructuredTool
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
#from langgraph.prebuilt import ToolExecutor

#### Load env variables & assign it to variables

In [3]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# CHROMA_API_KEY = os.getenv("CHROMA_API_KEY")

In [4]:
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")
print("Groq API key loaded successfully!")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file")
print("OpenAI API key loaded successfully!")

Groq API key loaded successfully!
OpenAI API key loaded successfully!


# Create & define tools and other helper functions that our agents can use

In [5]:
# Simple ArXiv Search Tool
def arxiv_search(query, max_results=10):
    """Search for papers on arXiv and return basic info.
    """
    arxiv_client = arxiv.Client()
    search = arxiv.Search(query=query, max_results=max_results)
    results = []
    for paper in arxiv_client.results(search):
        results.append({
            "id": paper.entry_id,
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            "url": paper.pdf_url,
            "abstract": paper.summary,
            "published": paper.published
        })
    return results

In [6]:
# Download ArXiv paper as pdf and parse text
def get_paper_and_parse(paper_id, max_char=10000):
    """Download a paper from arXiv by ID and extract text."""
    search = arxiv.Search(id_list=[paper_id])
    client = arxiv.Client()
    paper = next(client.results(search))
    response = requests.get(paper.pdf_url)
    pdf_file = BytesIO(response.content)
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text()

    return text[:max_char]

In [7]:
# Text Summarizer for the paper's contents
def paper_summarizer(text: str, 
                     groq_api_key: str=GROQ_API_KEY, 
                     max_char: int=30000):
    """Summarize text using Groq."""
    model = ChatGroq(api_key=groq_api_key, 
                     model_name="meta-llama/llama-4-maverick-17b-128e-instruct")
    prompt = f"Summarize this academic paper: {text[:max_char]}"
    return model.invoke(prompt).content

In [8]:
# Wrapper Function to create all tools above
def create_tools(groq_api_key: str,
                 max_search_results: int = 10,
                 max_char_download: int = 10000,
                 max_char_summarize: int=30000):
    """Create the tools for our agent."""

    tools = [
        Tool(
            name="search_arxiv",
            description="Search for academic papers on arXiv. Input is a search query.",
            func=lambda q: arxiv_search(q, max_search_results)
        ),
        Tool(
            name="download_paper",
            description="Download a paper from arXiv and extract its text. Input is a paper ID.",
            func=lambda id: get_paper_and_parse(id, max_char_download)
        ),
        Tool(
            name="summarize_text",
            description="Summarize a piece of text. Input is the text to summarize.",
            func=lambda t: paper_summarizer(t, groq_api_key, max_char_summarize)
        )
    ]

    return tools

# Agent Designs

Let's define the system prompt which we will use across all our implementations of this research agent

In [9]:
system_prompt = """
    You are a helpful AI assistant who is an expert research assistant.
    Your objective is to assist the user with comprehensive literature review by following the below steps as deemed appropriate:
    1. Take the user query 
    2. Search ArXiv for highly relevant papers to the query
    3. Fetch the metadata & contents of each paper from the search results
    4. Use the retrieved information to answer the user query by summarizing the contents of the paper into a dense block of relevant insights. You are expected to provide citations explicitly
    """

#### 1. Vanilla LLM Augmented

In [12]:
class VanillaLLMAugmented():
    def __init__(self, system_prompt, groq_api_key, openai_api_key, model="gpt-4o"):
        # Initialize various components of this agent
        self.llm = ChatOpenAI(model=model,
                              api_key=openai_api_key
                              )
        
        self.tools = create_tools(groq_api_key=groq_api_key)
        
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])
        
        self.agent = create_openai_tools_agent(llm=self.llm, 
                                               tools=self.tools,
                                               prompt=self.prompt)
        
        self.agent_executor = AgentExecutor(agent=self.agent, 
                                            tools=self.tools, 
                                            verbose=True)
        
    def run(self, query):
        return self.agent_executor.invoke({"input": query})

In [14]:
# Test the agent with a couple of sample queries

vanilla_llm_agent = VanillaLLMAugmented(system_prompt=system_prompt,
                                        groq_api_key=GROQ_API_KEY,
                                        openai_api_key=OPENAI_API_KEY,
                                        model="gpt-4o")

queries = [
    "What are the breakthroughs in the field of AI in the last decade?",
    "What are the latest developments in quantum computing?",
    "How has protein structure discovery changed over the years?"
    ]

responses = {}

for query in queries:
    response = vanilla_llm_agent.run(query)
    output = response["output"]
    responses[query]= output
    print(f"\n\nUSER QUERY: {query}")
    print(f"\nAGENT RESPONSE: {output}")



> Entering new AgentExecutor chain...

Invoking: `search_arxiv` with `Breakthroughs in AI in the last decade`


[{'id': 'http://arxiv.org/abs/2506.14640v1', 'title': 'Navigating the growing field of research on AI for software testing -- the taxonomy for AI-augmented software testing and an ontology-driven literature survey', 'authors': ['Ina K. Schieferdecker'], 'url': 'http://arxiv.org/pdf/2506.14640v1', 'abstract': "In industry, software testing is the primary method to verify and validate\nthe functionality, performance, security, usability, and so on, of\nsoftware-based systems. Test automation has gained increasing attention in\nindustry over the last decade, following decades of intense research into test\nautomation and model-based testing. However, designing, developing, maintaining\nand evolving test automation is a considerable effort. Meanwhile, AI's\nbreakthroughs in many engineering fields are opening up new perspectives for\nsoftware testing, for both manual and automa

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Project `proj_tReGmiDDku3ORHcipOnGveLG` does not have access to model `gpt-4o`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}